## Step 1: Environment Setup

In [ ]:
%%time
import os
import sys
import shutil
from pathlib import Path
import subprocess

# Configuration
REPO_INPUT = Path('/kaggle/input/gdsearch-repository')
WORKING_DIR = Path('/kaggle/working/GDSearch')
OUTPUT_DIR = Path('/kaggle/working/results')

print("="*80)
print("GDSearch Kaggle Environment Setup")
print("="*80)

# Check if repository exists
if not REPO_INPUT.exists():
    print("ERROR: Repository not found at /kaggle/input/gdsearch-repository")
    print("\nInstructions:")
    print("   1. Upload GDSearch repository as a Kaggle dataset")
    print("   2. Add dataset to this notebook")
    print("   3. Ensure it's mounted at /kaggle/input/gdsearch-repository")
    raise FileNotFoundError("GDSearch repository not found")

print(f"Repository found: {REPO_INPUT}")
print(f"Working directory: {WORKING_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

### Copy Repository to Working Directory

In [ ]:
%%time
print("Copying repository to working directory...")

# Remove existing working directory if present
if WORKING_DIR.exists():
    print(f"Removing existing {WORKING_DIR}")
    shutil.rmtree(WORKING_DIR)

# Copy repository
shutil.copytree(REPO_INPUT, WORKING_DIR, symlinks=False, ignore=None, dirs_exist_ok=True)
print(f"Repository copied to {WORKING_DIR}")

# Change to working directory
os.chdir(WORKING_DIR)
print(f"Current directory: {os.getcwd()}")

# Add to Python path
if str(WORKING_DIR) not in sys.path:
    sys.path.insert(0, str(WORKING_DIR))
    print(f"Added {WORKING_DIR} to Python path")

# Verify key files
key_files = ['run_all_kaggle.py', 'requirements.txt', 'src/__init__.py']
for file in key_files:
    if (WORKING_DIR / file).exists():
        print(f"Found {file}")
    else:
        print(f"Missing {file}")

### Resume from Previous Results (Optional)

**If you have previous Kaggle run results:**

1. Upload previous `gdsearch_results_complete.zip` as a Kaggle dataset
2. Name it `result` and add it to this notebook
3. It should mount at `/kaggle/input/result/`
4. Set `RESUME_ENABLED = True` in the cell below
5. Run the cell to copy previous results
6. Experiments will automatically skip completed runs!

In [ ]:
%%time
# ============================================================================
# RESUME FROM PREVIOUS KAGGLE RUN (Optional)
# ============================================================================
# If you uploaded previous results as a Kaggle dataset at /kaggle/input/result,
# this cell will copy them to the working directory so experiments can resume

import shutil
from pathlib import Path

PREVIOUS_RESULTS = Path('/kaggle/input/result')
RESUME_ENABLED = False  # Set to True if you want to resume from previous results

if RESUME_ENABLED and PREVIOUS_RESULTS.exists():
    print("="*80)
    print("RESUMING FROM PREVIOUS RESULTS")
    print("="*80)
    print(f"Source: {PREVIOUS_RESULTS}")
    print(f"Destination: {OUTPUT_DIR}")
    
    # Copy all previous results to working directory
    # This includes experiments/, checkpoints/, visualizations/, etc.
    if (PREVIOUS_RESULTS / 'results_full').exists():
        print("\nCopying previous results...")
        shutil.copytree(PREVIOUS_RESULTS / 'results_full', OUTPUT_DIR / 'results_full', 
                       dirs_exist_ok=True)
        
        # Count copied files
        copied_files = sum(1 for _ in (OUTPUT_DIR / 'results_full').rglob('*') if _.is_file())
        print(f"✓ Copied {copied_files} files from previous run")
        
        # Show what's available to resume
        experiments_dir = OUTPUT_DIR / 'results_full' / 'experiments'
        if experiments_dir.exists():
            completed_exp = [d.name for d in experiments_dir.iterdir() if d.is_dir()]
            print(f"\nCompleted experiments found: {', '.join(completed_exp)}")
        
        checkpoints_dir = OUTPUT_DIR / 'results_full' / 'checkpoints'
        if checkpoints_dir.exists():
            checkpoint_count = len(list(checkpoints_dir.glob('*.pt')))
            print(f"Checkpoints found: {checkpoint_count} model files")
        
        print("\n" + "="*80)
        print("RESUME SETUP COMPLETE")
        print("="*80)
        print("Run experiments with --resume flag to skip completed experiments")
        print("="*80)
    else:
        print(f"\nWARNING: {PREVIOUS_RESULTS / 'results_full'} not found")
        print("Expected structure: /kaggle/input/result/results_full/")
        print("Please check your dataset upload structure")
        
elif RESUME_ENABLED:
    print("="*80)
    print("RESUME REQUESTED BUT NO PREVIOUS RESULTS FOUND")
    print("="*80)
    print(f"Looking for: {PREVIOUS_RESULTS}")
    print("\nTo use resume:")
    print("1. Upload previous results as a Kaggle dataset")
    print("2. Add it to this notebook")
    print("3. Ensure it's mounted at /kaggle/input/result")
    print("4. Set RESUME_ENABLED = True in this cell")
    print("="*80)
else:
    print("Resume disabled (RESUME_ENABLED = False)")
    print("Starting fresh experiments from scratch")
    print("To enable resume: Set RESUME_ENABLED = True and upload previous results")

### Install Dependencies

### NumPy/Pandas Compatibility Check (CRITICAL)

In [ ]:
%%time
import sys
import subprocess

print("Checking NumPy/Pandas compatibility...")
print("="*80)

def run_pip(args):
    """Run pip command and capture output"""
    cmd = [sys.executable, '-m', 'pip'] + args
    print('>', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

# Check current NumPy version
try:
    import numpy as np
    numpy_version = np.__version__
    numpy_major = int(numpy_version.split('.')[0])
    print(f"NumPy: {numpy_version} (will use this version)")
except Exception as e:
    print(f"NumPy import failed: {e}")
    raise RuntimeError("NumPy must be available")

# Check if Pandas can import successfully
pandas_import_error = None
try:
    import pandas as pd
    pandas_version = pd.__version__
    print(f"Pandas: {pandas_version} - imports successfully!")
    pandas_ok = True
except ValueError as e:
    if "numpy.dtype size changed" in str(e):
        print(f"Pandas import failed: Binary incompatibility with NumPy {numpy_version}")
        print(f"   Error: {e}")
        pandas_import_error = e
        pandas_ok = False
    else:
        raise
except Exception as e:
    print(f"Pandas import failed: {e}")
    pandas_import_error = e
    pandas_ok = False

# Decision: Fix Pandas to match NumPy (keep NumPy version as-is)
if not pandas_ok:
    print("\n" + "="*80)
    print("FIXING: Reinstalling Pandas to match NumPy {numpy_version}")
    print("="*80)
    print(f"Best Practice: We keep NumPy {numpy_version} (Kaggle's optimized version)")
    print(f"Solution: Reinstall Pandas with --no-cache-dir to rebuild against current NumPy")
    print("\nThis ensures binary compatibility without downgrading platform packages.")
    print("="*80)
    
    # Reinstall pandas (will rebuild/redownload wheel compatible with current numpy)
    rc = run_pip(['install', '--force-reinstall', '--no-cache-dir', '--no-deps', 'pandas'])
    
    # Reinstall pandas dependencies that may have been skipped
    if rc == 0:
        print("\nReinstalling Pandas dependencies...")
        run_pip(['install', 'pandas'])  # This installs missing deps without forcing reinstall
    
    if rc == 0:
        print("\n" + "="*80)
        print("PANDAS REINSTALLED SUCCESSFULLY")
        print("="*80)
        print("\nCRITICAL: You MUST restart the kernel now!")
        print("   1. Click 'Runtime' → 'Restart runtime' (or Kernel → Restart)")
        print("   2. Re-run ALL cells from the beginning")
        print("   3. Pandas C-extensions will load correctly after restart")
        print("\nDO NOT PROCEED without restarting!")
        print("="*80)
    else:
        print("\nPandas reinstall failed - check error output above")
        raise RuntimeError("Pandas compatibility fix failed")
else:
    print("\nNumPy and Pandas are compatible - no action needed!")
    print(f"   Using NumPy {numpy_version} and Pandas {pandas_version}")
    print("="*80)

In [ ]:
%%time
print("Installing dependencies...")
print("="*80)

# Check if requirements_kaggle.txt exists, otherwise use requirements.txt
if (WORKING_DIR / 'kaggle' / 'requirements_kaggle.txt').exists():
    requirements_file = WORKING_DIR / 'kaggle' / 'requirements_kaggle.txt'
    print(f"Using Kaggle-specific requirements: {requirements_file}")
elif (WORKING_DIR / 'requirements.txt').exists():
    requirements_file = WORKING_DIR / 'requirements.txt'
    print(f"Using standard requirements: {requirements_file}")
    print("   NOTE: This may overwrite NumPy/Pandas. Prefer kaggle/requirements_kaggle.txt")
else:
    raise FileNotFoundError("No requirements file found")

# Install dependencies (suppress most output, show only errors)
print("\nInstalling packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Dependencies installed successfully")
    if result.stderr:
        # Show warnings but don't fail
        print("\nDependency warnings (usually safe to ignore):")
        # Filter out common non-critical warnings
        stderr_lines = result.stderr.split('\n')
        for line in stderr_lines[:20]:  # Show max 20 lines
            if line.strip() and 'incompatible' in line.lower():
                print(f"   {line}")
else:
    print("Dependency installation failed:")
    print(result.stderr)
    raise RuntimeError("Failed to install dependencies")

# Post-install verification
print("\nPost-install verification:")
critical_imports = [
    ('numpy', 'NumPy'),
    ('pandas', 'Pandas'),
    ('torch', 'PyTorch'),
    ('mlflow', 'MLflow'),
    ('optuna', 'Optuna'),
    ('transformers', 'Transformers'),
]

all_ok = True
for module_name, display_name in critical_imports:
    try:
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"   {display_name}: {version}")
    except Exception as e:
        print(f"   {display_name}: {e}")
        all_ok = False

if not all_ok:
    print("\nSome critical packages failed to import!")
    print("   Try manually installing the missing package(s) and re-running this cell.")
    raise RuntimeError("Critical import failures detected")

print("\n" + "="*80)
print("All critical dependencies verified!")
print("="*80)

In [ ]:
%%time
print("PRE-DOWNLOADING ALL DATASETS (CRITICAL for Kaggle time savings!)")
print("="*80)
print("This step downloads all datasets ONCE instead of repeatedly during experiments.")
print("Saves 30-60 minutes of Kaggle runtime!\n")

try:
    # Run the dataset download script
    result = subprocess.run(
        [sys.executable, 'download_datasets_kaggle.py'],
        capture_output=True,
        text=True,
        cwd=WORKING_DIR
    )
    
    # Show output
    print(result.stdout)
    if result.returncode != 0:
        print("\nDataset download warnings (non-critical):")
        print(result.stderr)
    
    print("\n" + "="*80)
    print("DATASET PRE-DOWNLOAD COMPLETE")
    print("="*80)
    print("All datasets cached and ready for experiments!")
    print("Experiments will run MUCH faster now.")
    print("="*80)
    
except Exception as e:
    print(f"\nDataset download failed: {e}")
    print("Experiments will download datasets on-demand (slower but still works)")
    print("="*80)

### Download Datasets

**Important:** Datasets will be downloaded automatically when experiments run, but you can pre-download them here to verify connectivity.

### Verify Environment

In [ ]:
print("Verifying environment...")
print("="*80)

# Check Python version
print(f"Python: {sys.version.split()[0]}")

# Check PyTorch and CUDA
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - CUDA version: {torch.version.cuda}")
    print(f"   - GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"     Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

# Check key dependencies
try:
    import numpy as np
    print(f"NumPy: {np.__version__}")
except ImportError as e:
    print(f"NumPy: {e}")

try:
    import pandas as pd
    print(f"Pandas: {pd.__version__}")
except ImportError as e:
    print(f"Pandas: {e}")

try:
    import matplotlib
    print(f"Matplotlib: {matplotlib.__version__}")
except ImportError as e:
    print(f"Matplotlib: {e}")

try:
    import tqdm
    print(f"tqdm: {tqdm.__version__}")
except ImportError as e:
    print(f"tqdm: {e}")

try:
    import mlflow
    print(f"MLflow: {mlflow.__version__}")
except ImportError as e:
    print(f"MLflow: {e}")

# Verify GDSearch modules
try:
    from src.core import optimizers
    print(f"GDSearch core modules imported")
except ImportError as e:
    print(f"GDSearch import error: {e}")

print("="*80)
print("Environment setup complete!")
print("="*80)

## Step 3: Run Experiments

## Scientific Methodology Notes (Gaps 35-64 Fixes)

This notebook implements comprehensive scientific methodology improvements for thesis defense:

### Statistical Improvements (Gaps 35, 37)
- **Data Soup Fix**: Results grouped by dataset (MNIST/CIFAR-10) to prevent mixing accuracies
- **Tuning Artifact Filter**: Optuna trial logs excluded from final analysis

### Loss Function Analysis (Gap 36)
- **Entropy Floor**: Label smoothing creates minimum loss > 0 (≈0.54 for 10 classes, smoothing=0.1)
- Convergence analysis accounts for this mathematical floor

### Convergence Analysis (Gaps 38, 32)
- **train_loss Default**: Convergence rate analyzed on training data (optimization), not test data (generalization)
- **O(1/√k) Rate Fitting**: Added root-sublinear rate for non-convex SGD bounds
- **Bonferroni Correction**: Multiple hypothesis testing correction applied

### Logging Improvements (Gaps 51-54)
- **Gradient Norms**: Logged before optimizer.step() for convergence analysis
- **Learning Rate**: Current LR logged each epoch
- **True Train Loss**: End-of-epoch evaluation (not running average)
- **Nesterov Support**: SGD_Nesterov optimizer added

### Fair Comparison (Gaps 47-48)
- **Weight Decay Standardized**: All optimizers use weight_decay=5e-4
- **Batch Ordering Randomized**: Each seed gets different data order

### Hessian Analysis (Gaps 60-61)
- **Renamed Metrics**: `trace_estimate` → `sum_top_k_eigenvalues` (NOT true trace)
- **Renamed Metrics**: `condition_number` → `top_k_spectral_ratio` (NOT true κ)

### Optimizer Implementation (Gap 43)
- **PyTorch Baseline**: Main benchmarks use torch.optim for performance
- **Custom Implementations**: Available via src.core.pytorch_optimizers wrappers
- Tests validate custom implementations match expected behavior

### Gradient Noise Estimation (NEW)
- **Configurable Sampling**: Use `--grad-noise-samples` to control sample count (default: 100)
- **Periodic Estimation**: Estimate gradient noise variance σ² every N epochs (configurable via `--grad-noise-every`)
- **Theoretical Bounds**: Enables validation of SGD convergence bounds with empirical noise measurements

### Dynamics Tracking (NEW)
- **Centralized Gradient Norms**: Uses `compute_gradient_norm()` helper for consistency
- **TrainingDynamicsTracker**: Available in specialized experiments (ablations, dynamics studies)
- **Parameter Snapshots**: Defaults to `track_params=False` to prevent OOM (343GB risk)
- **Disk-based Snapshots**: When enabled, uses incremental disk writes via `param_snapshot_dir`

### Robust Gradient Handling (NEW)
- **Adaptive Gradient Clipping (AGC)**: Per-layer gradient scaling via `--use-agc`
- **Global Clipping**: Norm-based clipping via `--gradient-clip-norm` (default: 1.0)
- **Trimmed-Mean Aggregation**: Robust to gradient outliers via `--use-trimmed-mean`
- **Heavy-Tail Monitoring**: Detects gradient distribution anomalies via `--monitor-heavy-tails`

In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION - FULL MODE FORCED
# =============================================================================

# ===== Option 1: Quick Test (5 minutes) =====
# Fast smoke test - 2 epochs, 3 seeds, MNIST only
# EXPERIMENT_MODE = 'quick'
# EXPERIMENTS = 'mnist'
# SEEDS = '42,123,456'
# EXTRA_ARGS = ['--ultra-quick', '--robust-gradients', '--grad-noise-every', '0']

# ===== Option 2: Proposal-Required Experiments (2-3 hours) =====
# All experiments needed for research proposal
# EXPERIMENT_MODE = 'proposal'
# EXPERIMENTS = 'mnist,2d,hyperparam_sensitivity,convergence_validation,theory_practice'
# SEEDS = '42,123,456,789,1011'
# EXTRA_ARGS = ['--kaggle-t4', '--time-budget', '3.0', '--robust-gradients', '--gradient-clip-norm', '1.0', '--grad-noise-every', '10', '--grad-noise-samples', '50']

# ===== Option 3: FULL MODE - FORCED (50 EPOCHS, NO QUICK MODE) =====
# RUNS ALL 31 EXPERIMENTS with FULL 50 EPOCHS (not quick mode's 20 epochs)
# 
# All 31 experiment types:
#   - mnist, cifar10, nlp, medical
#   - 2d, robustness, sam
#   - ablation, advanced_ablation, init_ablation, batch_ablation, lr_ablation, 
#     wd_ablation, scheduler_ablation, ablation_comprehensive
#   - optimizer_comparison, resnet, highdim
#   - hyperparam_sensitivity, convergence_validation
#   - 2d_visualization, dynamics_overhead, theory_practice
#   - cross_optimizer_dynamics, beta_sensitivity_training
#   - label_noise, saddle_escape
#   - hyperparameter_heatmaps, stochastic_2d_integrity, adam_adamw_comparison
# 
# TIMING ESTIMATE with 5 seeds + 50 epochs:
# - MNIST: ~6 hours (12 optimizers × 5 seeds × 50 epochs)
# - CIFAR10: ~3 hours (6 optimizers × 5 seeds × 50 epochs)
# - Other 29 experiments: ~3 hours
# - TOTAL: ~12 hours (will hit Kaggle limit)
# 
# NOTE: Some experiments may not complete due to 12h Kaggle limit
# Use --resume flag to continue in another session
# 
# NEW FLAGS:
# - --robust-gradients: AGC + trimmed-mean + heavy-tail monitoring
# - --gradient-clip-norm: Global gradient clipping (default: 1.0)
# - --grad-noise-every: Estimate gradient noise σ² every N epochs (0=disable)
# - --grad-noise-samples: Number of samples for noise estimation (default: 100)
# - --use-agc: Adaptive Gradient Clipping per-layer
# - --monitor-heavy-tails: Detect gradient outliers
EXPERIMENT_MODE = 'full'
EXPERIMENTS = 'all'  # All 31 experiment types
SEEDS = '42,123,456,789,1011'  # 5 seeds for statistical significance
EXTRA_ARGS = [
    '--kaggle-t4',  # T4 GPU optimizations
    '--time-budget', '11.8',  # 11.8 hours (leave 0.2h buffer for saving)
    '--robust-gradients',  # Enable robust gradient handling suite
    '--gradient-clip-norm', '1.0',  # Global gradient clipping
    '--grad-noise-every', '10',  # Estimate gradient noise variance every 10 epochs
    '--grad-noise-samples', '100',  # Use 100 samples for noise estimation
    '--use-agc',  # Adaptive Gradient Clipping
    '--monitor-heavy-tails'  # Detect gradient distribution outliers
]  # NO --quick flag = 50 epochs!

# ===== Option 4: Custom Configuration =====
# Customize experiments, seeds, and arguments
# EXPERIMENT_MODE = 'custom'
# EXPERIMENTS = 'mnist,cifar10,2d'  # Choose specific experiments
# SEEDS = '42,123,456'  # Minimum 3 seeds for statistics
# EXTRA_ARGS = ['--kaggle-t4', '--time-budget', '5.0', '--robust-gradients', '--grad-noise-every', '5']

# =============================================================================
# Results Directory
# =============================================================================
RESULTS_DIR = OUTPUT_DIR / f'results_{EXPERIMENT_MODE}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment Mode: {EXPERIMENT_MODE.upper()}")
print(f"Configuration: FULL MODE (50 EPOCHS) - NO QUICK MODE")
print(f"Experiments: {EXPERIMENTS}")
if EXPERIMENTS == 'all':
    print(f"   WARNING: 'all' runs 31 different experiment types!")
    print(f"   FULL MODE = 50 epochs per experiment (not quick mode's 20)")
    print(f"   With 5 seeds: ~12 hours (may hit Kaggle limit)")
    print(f"   Use --resume to continue incomplete experiments")
print(f"Seeds: {SEEDS} (5 seeds for statistical robustness)")
print(f"Extra Args: {' '.join(EXTRA_ARGS)}")
print(f"Expected Epochs: 50 (FULL MODE - no --quick flag)")
print(f"Robust Gradients: ENABLED (AGC + clipping for training stability)")
print(f"Gradient Noise Estimation: Every 10 epochs with 100 samples")
print(f"Results will be saved to: {RESULTS_DIR}")
print("="*80)


### Execute Experiments

In [ ]:
%%time
print("="*80)
print(f"Starting {EXPERIMENT_MODE.upper()} mode experiments")
print("="*80)

# Build command with optional --resume flag
cmd = [
    sys.executable,
    'run_all_kaggle.py',
    '--experiments', EXPERIMENTS,
    '--seeds', SEEDS,
    '--results-dir', str(RESULTS_DIR)
] + EXTRA_ARGS

# Add --resume flag if previous results were loaded
if RESUME_ENABLED and (OUTPUT_DIR / 'results_full').exists():
    cmd.append('--resume')
    print("🔄 RESUME MODE ENABLED - Will skip completed experiments")
    print("="*80)

print("Command:")
print(' '.join(cmd))
print("\n" + "="*80)

# Run experiments
import time
start_time = time.time()

try:
    result = subprocess.run(
        cmd,
        cwd=WORKING_DIR,
        check=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        universal_newlines=True
    )
    
    # Print output
    print(result.stdout)
    
    elapsed = time.time() - start_time
    print("\n" + "="*80)
    print(f"Experiments completed successfully!")
    print(f"Total time: {elapsed/3600:.2f} hours ({elapsed/60:.1f} minutes)")
    print("="*80)
    
except subprocess.CalledProcessError as e:
    elapsed = time.time() - start_time
    print(f"\nExperiments failed after {elapsed/60:.1f} minutes")
    print("Error output:")
    print(e.stdout)
    raise

## Step 4: Results Analysis

### List Generated Results

In [ ]:
import os
from pathlib import Path

print("Generated Results:")
print("="*80)

# List all result directories
result_dirs = [
    'experiments',
    '2d_optimization',
    'beta_sensitivity',
    'hyperparameter_sensitivity',
    'theory_practice',
    'visualizations',
    'analysis',
    'reports'
]

for dir_name in result_dirs:
    dir_path = RESULTS_DIR / dir_name
    if dir_path.exists():
        file_count = sum(1 for _ in dir_path.rglob('*') if _.is_file())
        print(f"{dir_name}: {file_count} files")
        
        # Show first few files
        files = sorted(dir_path.rglob('*.csv'))[:5]
        if files:
            for f in files:
                rel_path = f.relative_to(RESULTS_DIR)
                print(f"   - {rel_path}")
            if len(list(dir_path.rglob('*.csv'))) > 5:
                print(f"   ... and {len(list(dir_path.rglob('*.csv'))) - 5} more CSV files")
    else:
        print(f"{dir_name}: not found")

print("="*80)

### Quick Results Preview

In [ ]:
import pandas as pd
import glob

print("Quick Results Preview:")
print("="*80)

# Find MNIST results
mnist_csvs = list((RESULTS_DIR / 'experiments' / 'mnist').glob('*.csv'))

if mnist_csvs:
    print(f"\nFound {len(mnist_csvs)} MNIST result files\n")
    
    # Load and display summary
    results = []
    for csv in mnist_csvs[:10]:  # Show first 10
        df = pd.read_csv(csv)
        if len(df) > 0:
            final_row = df.iloc[-1]
            results.append({
                'file': csv.name,
                'epochs': len(df),
                'final_train_loss': final_row.get('train_loss', 'N/A'),
                'final_test_acc': final_row.get('test_acc', 'N/A'),
                'final_grad_norm': final_row.get('grad_norm', 'N/A')
            })
    
    if results:
        summary_df = pd.DataFrame(results)
        print(summary_df.to_string(index=False))
        
        # Check for grad_norm column
        if 'final_grad_norm' in summary_df.columns:
            has_grad_norm = summary_df['final_grad_norm'] != 'N/A'
            if has_grad_norm.all():
                print("\nAll results include gradient norm tracking!")
            else:
                print("\nSome results missing gradient norm")
else:
    print("No MNIST results found")

print("\n" + "="*80)

### Display Visualizations

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
import glob

print("Visualizations:")
print("="*80)

# Find visualization PNGs
viz_dir = RESULTS_DIR / 'visualizations' / 'static'
if viz_dir.exists():
    png_files = sorted(viz_dir.rglob('*.png'))[:5]  # Show first 5
    
    if png_files:
        for png in png_files:
            print(f"\n{png.name}")
            try:
                display(Image(filename=str(png), width=800))
            except Exception as e:
                print(f"   Could not display: {e}")
    else:
        print("No visualization PNGs found")
else:
    print("Visualization directory not found")

print("\n" + "="*80)

## Step 5: Save Results for Download

In [ ]:
%%time
print("Preparing results for download...")
print("="*80)

# Create archive
import shutil
archive_name = f'gdsearch_results_{EXPERIMENT_MODE}'
archive_path = OUTPUT_DIR / archive_name

print(f"Creating archive: {archive_name}.zip")
shutil.make_archive(str(archive_path), 'zip', RESULTS_DIR)

# Get archive size
archive_file = f"{archive_path}.zip"
size_mb = os.path.getsize(archive_file) / (1024 * 1024)
print(f"Archive created: {size_mb:.2f} MB")

# Summary
print("\n" + "="*80)
print("Results saved to:")
print(f"   - Directory: {RESULTS_DIR}")
print(f"   - Archive: {archive_file}")
print("\nTo download:")
print("   1. Check the 'Output' tab in Kaggle")
print(f"   2. Download {archive_name}.zip")
print("   3. Extract and analyze locally")
print("="*80)

## Step 6: Experiment Summary Report

In [ ]:
# Check for auto-generated summary report
summary_report = RESULTS_DIR / 'reports' / '00_EXPERIMENT_SUMMARY.md'

print("Experiment Summary:")
print("="*80)

if summary_report.exists():
    with open(summary_report, 'r') as f:
        print(f.read())
else:
    print("Summary report not found")
    print("\nManual Summary:")
    print(f"- Mode: {EXPERIMENT_MODE}")
    print(f"- Experiments: {EXPERIMENTS}")
    print(f"- Seeds: {SEEDS}")
    print(f"- Results directory: {RESULTS_DIR}")

print("\n" + "="*80)

---

## Completion Checklist

After running this notebook, verify:

- [ ] Environment setup completed without errors
- [ ] Quick validation test passed
- [ ] Experiments ran successfully
- [ ] Results generated in expected directories
- [ ] CSV files contain required columns (grad_norm, test_acc, etc.)
- [ ] Visualizations generated (if applicable)
- [ ] Results archive created for download

---

## Troubleshooting

### Common Issues:

**1. Repository not found**
```python
# Check dataset mounting:
!ls /kaggle/input/
```

**2. Out of Memory (OOM)**
```python
# Reduce batch size or use ultra-quick mode
EXTRA_ARGS = ['--ultra-quick', '--batch-size', '32']
```

**3. Time limit exceeded**
```python
# Reduce time budget or number of seeds
SEEDS = '42,123,456'  # Use fewer seeds
EXTRA_ARGS = ['--time-budget', '3.0']  # Lower budget
```

**4. Missing dependencies**
```python
# Manually install missing package
!pip install <package-name>
```

---

## Additional Resources

- **Documentation:** See `README.md` in repository
- **Proposal Compliance:** See `docs/PROPOSAL_COMPLIANCE_CHECKLIST.md`
- **Configuration Schema:** See `configs/config_schema.json`

---

**Generated by GDSearch Kaggle Runner**  
*Last Updated: December 24, 2025*

## Quick Download Results (Add this link)

In [ ]:
# ============================================================================
# DOWNLOAD ALL RESULTS (Everything including checkpoints)
# ============================================================================

from IPython.display import FileLink
import shutil
import os

print("Creating downloadable archive of ALL RESULTS...")
print("="*80)

# Archive the entire results directory (including checkpoints)
archive_path = '/kaggle/working/gdsearch_results_complete'
shutil.make_archive(archive_path, 'zip', RESULTS_DIR)

# Get archive size
archive_size_mb = os.path.getsize(f'{archive_path}.zip') / (1024**2)

# Count files
file_count = sum(1 for _ in RESULTS_DIR.rglob('*') if _.is_file())

print("\n" + "="*80)
print("✅ COMPLETE RESULTS ARCHIVE CREATED")
print("="*80)
print(f"📦 Archive: {archive_path}.zip")
print(f"📊 Size: {archive_size_mb:.2f} MB")
print(f"📁 Files: {file_count}")
print(f"📂 Includes: experiments, checkpoints, visualizations, reports, analysis")
print(f"\n📥 Download from Kaggle Output tab or click link below:")
print("="*80)

# Display download link
FileLink(f'{archive_path}.zip')

## Quick Download Results

In [ ]:
# QUICK DOWNLOAD: Click the folder icon and download results manually
# OR use this command to create a downloadable archive:

#from IPython.display import FileLink
#import shutil

#print("Creating downloadable results archive...")
#archive_path = '/kaggle/working/results_download'
#shutil.make_archive(archive_path, 'zip', RESULTS_DIR)

#print(f"\nResults archived: {archive_path}.zip")
#print(f"Size: {os.path.getsize(f'{archive_path}.zip') / (1024**2):.2f} MB")
#print("\n📥 Download from Output tab or use link below:")

# Create download link
#FileLink(f'{archive_path}.zip')